数据生成

In [1]:
# data_generator.py
import os
import shutil
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

# ================= 配置区域 =================
# 路径配置
DATASET_ROOT = "./dataset_dual_RA" 
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")

# 真实数据映射 (请根据你的实际路径修改)
FILE_MAPPING = {
    "E:/Project/AMR-Benchmark/BPSK-03-24-25-10h52m32s937.iq": "BPSK",
    "E:/Project/AMR-Benchmark/QAM64-03-24-25-10h51m51s805.iq": "QAM64",
    "E:/Project/AMR-Benchmark/QAM16-03-24-25-10h52m09s762.iq": "QAM16",
    "E:/Project/AMR-Benchmark/PSK8-03-24-25-10h54m48s14.iq": "PSK8",
    "E:/Project/AMR-Benchmark/QPSK-03-24-25-10h53m20s46.iq": "QPSK",
    "E:/Project/AMR-Benchmark/FSK2DEV625KHZ-03-24-25-11h08m50s462.iq": "FSK2"
}

CLASS_NAMES = ["BPSK", "QPSK", "PSK8", "QAM16", "QAM64", "FSK2"]
SAMPLES_PER_IMAGE = 10240
IMAGE_SIZE = 64
TOTAL_SIM_SAMPLES_PER_CLASS = 800
TRAIN_VAL_RATIO = 0.8
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# ================= 1. 数学与图像工具 =================

def fast_hist2d(x, y, bins=64, range_bounds=((-1, 1), (-1, 1))):
    x_min, x_max = range_bounds[0]
    y_min, y_max = range_bounds[1]
    x = torch.clamp(x, x_min, x_max)
    y = torch.clamp(y, y_min, y_max)
    x_idx = ((x - x_min) / (x_max - x_min) * (bins - 1)).long()
    y_idx = ((y - y_min) / (y_max - y_min) * (bins - 1)).long()
    flat_idx = y_idx * bins + x_idx
    hist = torch.bincount(flat_idx, minlength=bins*bins)
    return hist.view(bins, bins).float()

def generate_manifold_heatmap(iq_complex_data, img_size=64):
    """计算 R 通道 和 A 通道, Output: (2, H, W)"""
    if iq_complex_data.dim() == 1:
        iq_complex_data = iq_complex_data.unsqueeze(0)
    iq_data = iq_complex_data.squeeze()
    if iq_data.numel() < 100: return torch.zeros(2, img_size, img_size)

    scale = torch.sqrt(torch.mean(iq_data.real**2 + iq_data.imag**2)) + 1e-9
    norm_iq = iq_data / (3.0 * scale)
    
    real = norm_iq.real
    imag = norm_iq.imag
    phase = torch.angle(norm_iq) / np.pi 
    mag = torch.abs(norm_iq)
    
    hist_r = fast_hist2d(real, imag, bins=img_size, range_bounds=((-1.2, 1.2), (-1.2, 1.2)))
    hist_a = fast_hist2d(phase, mag, bins=img_size, range_bounds=((-1.0, 1.0), (0.0, 1.5)))
    
    def process(h):
        h = torch.log1p(h)
        return (h - h.min()) / (h.max() - h.min() + 1e-8)
        
    img_tensor = torch.stack([process(hist_r), process(hist_a)], dim=0) 
    return torch.flip(img_tensor, [1]) 

def save_tensor_as_dual_channel_png(tensor, path):
    """保存为 RGB PNG (R=Ch1, G=Ch2, B=0)"""
    tensor = tensor.detach().cpu()
    rgb_tensor = torch.zeros(3, tensor.shape[1], tensor.shape[2])
    rgb_tensor[0] = tensor[0] # Red = R Channel
    rgb_tensor[1] = tensor[1] # Green = A Channel
    arr = (rgb_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    Image.fromarray(arr, mode='RGB').save(path)

# ================= 2. 信号仿真核心 =================

def generate_rrc_filter(N_taps, beta, sps):
    t = torch.arange(-(N_taps - 1) // 2, (N_taps - 1) // 2 + 1, dtype=torch.float32) / sps
    mid_idx = (N_taps - 1) // 2
    t[mid_idx] = 1e-8
    num = torch.sin(np.pi * t * (1 - beta)) + 4 * beta * t * torch.cos(np.pi * t * (1 + beta))
    den = np.pi * t * (1 - (4 * beta * t)**2)
    coeffs = num / den
    coeffs[mid_idx] = 1 - beta + (4 * beta / np.pi)
    return coeffs / torch.sqrt(torch.sum(coeffs**2))

def apply_channel_effects(sig, snr_db):
    N = len(sig)
    freq_offset = np.random.uniform(-50, 50) 
    t = torch.arange(N, dtype=torch.float32) / 80e3
    phase_offset = np.random.uniform(0, 2*np.pi)
    rotation = torch.exp(1j * (2 * np.pi * freq_offset * t + phase_offset))
    sig = sig * rotation
    
    phase_jitter = torch.randn(N) * 0.005 
    sig = sig * torch.exp(1j * phase_jitter)
    
    g_imbalance = np.random.uniform(0.98, 1.02)
    p_imbalance = np.random.uniform(-1, 1) * np.pi/180
    real = sig.real * g_imbalance
    imag = sig.imag * np.cos(p_imbalance) - sig.real * np.sin(p_imbalance)
    sig = torch.complex(real, imag)
    
    sig_power = torch.mean(torch.abs(sig)**2)
    real_snr = snr_db + 10 
    noise_power = sig_power / (10**(real_snr/10.0))
    noise = (torch.randn_like(sig) + 1j*torch.randn_like(sig)) * torch.sqrt(noise_power/2)
    return sig + noise

def generate_sim_signal(mod_type, N, snr_db):
    Fs, Rs = 80e3, 20e3
    sps = int(Fs / Rs)
    num_symbols = int(N / sps) + 50
    
    if mod_type == "BPSK":
        syms = 2*torch.randint(0, 2, (num_symbols,)) - 1 + 0j
    elif mod_type == "QPSK":
        phase = np.pi/4 + torch.randint(0, 4, (num_symbols,)) * np.pi/2
        syms = torch.exp(1j * phase)
    elif mod_type == "PSK8":
        phase = torch.randint(0, 8, (num_symbols,)) * 2*np.pi/8
        syms = torch.exp(1j * phase)
    elif mod_type == "QAM16":
        idx = torch.randint(0, 16, (num_symbols,))
        syms = ((idx // 4) * 2 - 3) + 1j*((idx % 4) * 2 - 3)
    elif mod_type == "QAM64":
        idx = torch.randint(0, 64, (num_symbols,))
        syms = ((idx // 8) * 2 - 7) + 1j*((idx % 8) * 2 - 7)
    elif mod_type == "FSK2":
        bits = torch.randint(0, 2, (N + 100,))
        f_dev = np.random.uniform(500, 6000) 
        freqs = (bits * 2 - 1) * f_dev
        phase = 2 * np.pi * torch.cumsum(freqs/Fs, dim=0)
        sig = torch.exp(1j * phase)[:N]
        return apply_channel_effects(sig, snr_db)

    upsampled = torch.zeros(num_symbols * sps, dtype=torch.complex64)
    upsampled[::sps] = syms
    taps = 10 * sps + 1
    h = generate_rrc_filter(taps, 0.35, sps)
    real = F.conv1d(upsampled.real.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    imag = F.conv1d(upsampled.imag.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    sig = (real + 1j*imag).squeeze()
    if len(sig) > N: sig = sig[:N]
    else: sig = F.pad(sig, (0, N - len(sig)))
    return apply_channel_effects(sig, snr_db)

# ================= 3. 执行生成 =================

if __name__ == "__main__":
    if os.path.exists(DATASET_ROOT):
        print("Cleaning old dataset...")
        shutil.rmtree(DATASET_ROOT)
    
    print("===== 1. Generating Simulated Data (R+A) =====")
    for cls_name in CLASS_NAMES:
        os.makedirs(os.path.join(SIM_TRAIN_DIR, cls_name), exist_ok=True)
        os.makedirs(os.path.join(SIM_VAL_DIR, cls_name), exist_ok=True)
        print(f"Generating {cls_name}...")
        for i in tqdm(range(TOTAL_SIM_SAMPLES_PER_CLASS)):
            target_dir = SIM_TRAIN_DIR if np.random.rand() < TRAIN_VAL_RATIO else SIM_VAL_DIR
            target_dir = os.path.join(target_dir, cls_name)
            
            # 随机 SNR (-5 ~ 25dB)
            snr = np.random.uniform(-5, 25)
            sig = generate_sim_signal(cls_name, SAMPLES_PER_IMAGE, snr)
            heatmap = generate_manifold_heatmap(sig.to(DEVICE), IMAGE_SIZE)
            
            # 文件名包含 SNR 信息，格式: Class_Index_SNR.png
            save_path = os.path.join(target_dir, f"{cls_name}_{i:04d}_{snr:.1f}.png")
            save_tensor_as_dual_channel_png(heatmap, save_path)

    print("\n===== 2. Processing Real Data (R+A) =====")
    for file_path, label in FILE_MAPPING.items():
        if not os.path.exists(file_path): 
            print(f"Skip: {file_path}")
            continue
        print(f"Processing {label}...")
        save_dir = os.path.join(REAL_TEST_DIR, label)
        os.makedirs(save_dir, exist_ok=True)
        try:
            raw = np.fromfile(file_path, dtype=np.int16)
            if len(raw) % 2 != 0: raw = raw[:-1]
            I = raw[::2].astype(np.float32)
            Q = raw[1::2].astype(np.float32)
            iq_complex = (I + 1j * Q).astype(np.complex64) / 32767.0
            
            num_slices = len(iq_complex) // SAMPLES_PER_IMAGE
            for i in range(num_slices):
                chunk = iq_complex[i*SAMPLES_PER_IMAGE : (i+1)*SAMPLES_PER_IMAGE]
                chunk_tensor = torch.from_numpy(chunk).to(DEVICE)
                heatmap = generate_manifold_heatmap(chunk_tensor, IMAGE_SIZE)
                # 真实数据没有 SNR 标签，标记为 'real'
                save_tensor_as_dual_channel_png(heatmap, os.path.join(save_dir, f"real_{i:04d}_unknown.png"))
        except Exception as e:
            print(f"Error: {e}")
            
    print("\nData generation complete!")

d:\anaconda3\envs\pytorch1_12_1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cleaning old dataset...
===== 1. Generating Simulated Data (R+A) =====
Generating BPSK...


100%|██████████| 800/800 [00:03<00:00, 226.39it/s]


Generating QPSK...


100%|██████████| 800/800 [00:02<00:00, 314.29it/s]


Generating PSK8...


... (4 outputs left out)

  0%|          | 0/800 [00:00<?, ?it/s]C:\Users\CPC-ML\AppData\Local\Temp\ipykernel_6384\3988091612.py:134: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  syms = ((idx // 8) * 2 - 7) + 1j*((idx % 8) * 2 - 7)
100%|██████████| 800/800 [00:03<00:00, 233.12it/s]


Generating FSK2...


100%|██████████| 800/800 [00:02<00:00, 329.00it/s]



===== 2. Processing Real Data (R+A) =====
Processing BPSK...
Processing QAM64...
Processing QAM16...
Processing PSK8...
Processing QPSK...
Processing FSK2...

Data generation complete!


In [2]:
# main_train.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib
matplotlib.use('Agg') # 确保后台运行不弹窗
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import numpy as np

# ================= 配置区域 =================
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
DATASET_ROOT = "./dataset_dual_RA" 
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")
RESULTS_DIR = "./training_results" # 结果保存目录

CLASS_NAMES = ["BPSK", "QPSK", "PSK8", "QAM16", "QAM64", "FSK2"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 64
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 15

# 创建结果目录
os.makedirs(RESULTS_DIR, exist_ok=True)

# ================= 1. 数据集定义 (带 SNR 解析) =================
class InMemoryDataset(Dataset):
    def __init__(self, root_dir, has_snr=False):
        self.data = []
        self.labels = []
        self.snrs = [] # 存储每个样本的 SNR
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), 
            transforms.ToTensor()
        ])
        
        if not os.path.exists(root_dir):
            print(f"Path not found: {root_dir}")
            return
            
        print(f"Loading {root_dir} into RAM...")
        files = []
        for cls_name in CLASS_NAMES:
            d = Path(root_dir) / cls_name
            if d.exists(): 
                # 收集所有 png 文件
                files.extend([(p, CLASS_NAMES.index(cls_name)) for p in d.glob("*.png")])
        
        with tqdm(total=len(files), unit="img") as pbar:
            for p, label in files:
                try:
                    # 1. 解析 SNR (文件名格式: Class_Index_SNR.png)
                    snr = -999.0 # 默认值 (真实数据)
                    if has_snr:
                        try:
                            # 取倒数第一个部分，去掉 .png
                            fname = p.stem # e.g. "BPSK_0012_15.5"
                            snr_str = fname.split('_')[-1]
                            snr = float(snr_str)
                        except: pass
                    
                    # 2. 读取图片 (RGB)
                    with Image.open(str(p)).convert("RGB") as img:
                        tensor = self.transform(img)
                        # 只取前两通道 R+A
                        self.data.append(tensor[:2, :, :]) 
                        self.labels.append(label)
                        self.snrs.append(snr)
                except Exception as e:
                    pass
                pbar.update(1)

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): 
        # 返回数据、标签、以及该样本的 SNR
        return self.data[idx], self.labels[idx], self.snrs[idx]

# ================= 2. 模型定义 =================
class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super(TinyCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

# ================= 3. 辅助绘图函数 =================
def plot_training_curves(history, save_path):
    """绘制 Loss 和 Accuracy 曲线"""
    epochs = range(1, len(history['train_acc']) + 1)
    
    plt.figure(figsize=(12, 5))
    
    # Loss Curve
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    # plt.plot(epochs, history['val_loss'], 'r-', label='Val Loss') # 验证集 Loss 可选
    plt.title('Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Accuracy Curve
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    plt.plot(epochs, history['val_acc'], 'r-', label='Val Acc')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_confusion_matrix(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ================= 4. 主训练逻辑 =================
def main():
    # --- 加载数据 ---
    print("Initializing Datasets...")
    # 训练集不需要 SNR 信息
    train_ds = InMemoryDataset(SIM_TRAIN_DIR, has_snr=False) 
    # 验证集需要 SNR 信息做分析
    val_ds = InMemoryDataset(SIM_VAL_DIR, has_snr=True) 
    # 真实测试集不需要 SNR (或者说是未知的)
    real_ds = InMemoryDataset(REAL_TEST_DIR, has_snr=False)
    
    if len(train_ds) == 0:
        print("No data found! Run data_generator.py first.")
        return

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # --- 模型准备 ---
    model = TinyCNN(NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
    
    print(f"\nStarting Training for {NUM_EPOCHS} epochs...")
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for imgs, labels, _ in train_loader: # 忽略 SNR
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                _, preds = outputs.max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total += labels.size(0)
        
        val_acc = val_correct / val_total
        
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} | Val Acc: {val_acc:.4f}")

    # 保存曲线
    plot_training_curves(history, os.path.join(RESULTS_DIR, "training_curves.png"))
    
    # --- 5. 详细评估 (Per-SNR Evaluation) ---
    print("\nRunning Detailed Evaluation on Validation Set...")
    model.eval()
    
    # 存储所有验证结果
    val_results = {'true': [], 'pred': [], 'snr': []}
    
    with torch.no_grad():
        for imgs, labels, snrs in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            _, preds = outputs.max(1)
            
            val_results['true'].extend(labels.cpu().numpy())
            val_results['pred'].extend(preds.cpu().numpy())
            val_results['snr'].extend(snrs) # snrs 是 list
            
    # 将结果转为 numpy
    y_true = np.array(val_results['true'])
    y_pred = np.array(val_results['pred'])
    snrs = np.array(val_results['snr'])
    
    # 定义 SNR 区间 (Bins)
    snr_bins = [(-5, 0), (0, 5), (5, 10), (10, 15), (15, 20), (20, 30)]
    
    print("Generating Per-SNR Confusion Matrices...")
    for (low, high) in snr_bins:
        # 找到属于该区间的索引
        mask = (snrs >= low) & (snrs < high)
        if np.sum(mask) == 0: continue
        
        subset_true = y_true[mask]
        subset_pred = y_pred[mask]
        subset_acc = np.mean(subset_true == subset_pred)
        
        title = f"SNR Range {low}dB to {high}dB (Acc: {subset_acc:.2%})"
        filename = os.path.join(RESULTS_DIR, f"cm_snr_{low}_{high}.png")
        
        plot_confusion_matrix(subset_true, subset_pred, title, filename)
        print(f"  -> Saved {filename} (Acc: {subset_acc:.2%})")

    # --- 6. 真实数据测试 ---
    if len(real_ds) > 0:
        print("\nRunning Real Data Test...")
        real_loader = DataLoader(real_ds, batch_size=BATCH_SIZE, shuffle=False)
        real_true = []
        real_pred = []
        
        with torch.no_grad():
            for imgs, labels, _ in real_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                _, preds = outputs.max(1)
                real_true.extend(labels.cpu().numpy())
                real_pred.extend(preds.cpu().numpy())
        
        real_acc = np.mean(np.array(real_true) == np.array(real_pred))
        print(f"Real Data Accuracy: {real_acc:.4f}")
        
        plot_confusion_matrix(real_true, real_pred, 
                             f"Real Data Confusion Matrix (Acc: {real_acc:.2%})",
                             os.path.join(RESULTS_DIR, "cm_real_data.png"))
        print(f"Results saved to {RESULTS_DIR}")

if __name__ == "__main__":
    main()

Initializing Datasets...
Loading ./dataset_dual_RA\sim_train into RAM...


100%|██████████| 3783/3783 [00:13<00:00, 273.35img/s]


Loading ./dataset_dual_RA\sim_val into RAM...


100%|██████████| 1017/1017 [00:03<00:00, 292.80img/s]


Loading ./dataset_dual_RA\real_test into RAM...


100%|██████████| 29292/29292 [01:48<00:00, 268.92img/s]



Starting Training for 15 epochs...
Epoch 1/15 | Loss: 1.1412 | Train Acc: 0.5279 | Val Acc: 0.7748
Epoch 2/15 | Loss: 0.5056 | Train Acc: 0.7978 | Val Acc: 0.9174
Epoch 3/15 | Loss: 0.3275 | Train Acc: 0.8768 | Val Acc: 0.9184
Epoch 4/15 | Loss: 0.2776 | Train Acc: 0.8980 | Val Acc: 0.9174
Epoch 5/15 | Loss: 0.2444 | Train Acc: 0.9054 | Val Acc: 0.9282
Epoch 6/15 | Loss: 0.2259 | Train Acc: 0.9093 | Val Acc: 0.9282
Epoch 7/15 | Loss: 0.2130 | Train Acc: 0.9186 | Val Acc: 0.9272
Epoch 8/15 | Loss: 0.2091 | Train Acc: 0.9188 | Val Acc: 0.9282
Epoch 9/15 | Loss: 0.1864 | Train Acc: 0.9252 | Val Acc: 0.9331
Epoch 10/15 | Loss: 0.1758 | Train Acc: 0.9281 | Val Acc: 0.9410
Epoch 11/15 | Loss: 0.1651 | Train Acc: 0.9337 | Val Acc: 0.9400
Epoch 12/15 | Loss: 0.1702 | Train Acc: 0.9281 | Val Acc: 0.9331
Epoch 13/15 | Loss: 0.1732 | Train Acc: 0.9270 | Val Acc: 0.9361
Epoch 14/15 | Loss: 0.1669 | Train Acc: 0.9289 | Val Acc: 0.9420
Epoch 15/15 | Loss: 0.1594 | Train Acc: 0.9307 | Val Acc: 0.944

In [14]:
# main_train.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib
matplotlib.use('Agg') # 确保后台运行不弹窗
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import numpy as np

# ================= 配置区域 =================
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
DATASET_ROOT = "./dataset_dual_RA" 
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")
RESULTS_DIR = "./training_results" 

CLASS_NAMES = ["BPSK", "QPSK", "PSK8", "QAM16", "QAM64", "FSK2"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 64
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 15

os.makedirs(RESULTS_DIR, exist_ok=True)

# ================= 1. 数据集定义 =================
class InMemoryDataset(Dataset):
    def __init__(self, root_dir, has_snr=False):
        self.data = []
        self.labels = []
        self.snrs = [] 
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), 
            transforms.ToTensor()
        ])
        
        if not os.path.exists(root_dir):
            print(f"Path not found: {root_dir}")
            return
            
        print(f"Loading {root_dir} into RAM...")
        files = []
        for cls_name in CLASS_NAMES:
            d = Path(root_dir) / cls_name
            if d.exists(): 
                files.extend([(p, CLASS_NAMES.index(cls_name)) for p in d.glob("*.png")])
        
        with tqdm(total=len(files), unit="img") as pbar:
            for p, label in files:
                try:
                    snr = -999.0
                    if has_snr:
                        try:
                            fname = p.stem 
                            snr_str = fname.split('_')[-1]
                            snr = float(snr_str)
                        except: pass
                    
                    with Image.open(str(p)).convert("RGB") as img:
                        tensor = self.transform(img)
                        self.data.append(tensor[:2, :, :]) # 只取 R+A
                        self.labels.append(label)
                        self.snrs.append(snr)
                except Exception as e:
                    pass
                pbar.update(1)

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): 
        return self.data[idx], self.labels[idx], self.snrs[idx]

# ================= 2. 模型定义 =================
class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super(TinyCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

# ================= 3. 辅助绘图函数 =================
def plot_training_curves(history, save_path):
    """绘制 Loss 和 Accuracy 曲线 (含验证集 Loss)"""
    epochs = range(1, len(history['train_acc']) + 1)
    
    plt.figure(figsize=(12, 5))
    
    # Loss Curve
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    # [新增] 绘制验证集 Loss
    plt.plot(epochs, history['val_loss'], 'r-', label='Val Loss') 
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    # Accuracy Curve
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    plt.plot(epochs, history['val_acc'], 'r-', label='Val Acc')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_confusion_matrix(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ================= 4. 主训练逻辑 =================
def main():
    # --- 加载数据 ---
    print("Initializing Datasets...")
    train_ds = InMemoryDataset(SIM_TRAIN_DIR, has_snr=False) 
    val_ds = InMemoryDataset(SIM_VAL_DIR, has_snr=True) 
    real_ds = InMemoryDataset(REAL_TEST_DIR, has_snr=False)
    
    if len(train_ds) == 0:
        print("No data found! Run data_generator.py first.")
        return

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # --- 模型准备 ---
    model = TinyCNN(NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    
    # [新增] 记录 val_loss
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    print(f"\nStarting Training for {NUM_EPOCHS} epochs...")
    for epoch in range(NUM_EPOCHS):
        # --- Training ---
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for imgs, labels, _ in train_loader: 
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        
        # --- Validation (计算 Loss 和 Acc) ---
        model.eval()
        val_running_loss = 0.0 # [新增]
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                
                # [新增] 计算验证 Loss
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * imgs.size(0)
                
                _, preds = outputs.max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total += labels.size(0)
        
        val_loss = val_running_loss / val_total # [新增]
        val_acc = val_correct / val_total
        
        # 记录历史
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)
        history['val_loss'].append(val_loss) # [新增]
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
              f"Loss: {epoch_loss:.4f} / {val_loss:.4f} | "  # 显示 Train / Val Loss
              f"Acc: {epoch_acc:.4f} / {val_acc:.4f}")      # 显示 Train / Val Acc

    # 保存曲线
    plot_training_curves(history, os.path.join(RESULTS_DIR, "training_curves.png"))
    
    # --- 5. 详细评估 (Per-SNR) ---
    print("\nRunning Detailed Evaluation on Validation Set...")
    model.eval()
    
    val_results = {'true': [], 'pred': [], 'snr': []}
    
    with torch.no_grad():
        for imgs, labels, snrs in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            _, preds = outputs.max(1)
            
            val_results['true'].extend(labels.cpu().numpy())
            val_results['pred'].extend(preds.cpu().numpy())
            val_results['snr'].extend(snrs)
            
    y_true = np.array(val_results['true'])
    y_pred = np.array(val_results['pred'])
    snrs = np.array(val_results['snr'])
    
    snr_bins = [(-5, 0), (0, 5), (5, 10), (10, 15), (15, 20), (20, 30)]
    
    print("Generating Per-SNR Confusion Matrices...")
    for (low, high) in snr_bins:
        mask = (snrs >= low) & (snrs < high)
        if np.sum(mask) == 0: continue
        
        subset_true = y_true[mask]
        subset_pred = y_pred[mask]
        subset_acc = np.mean(subset_true == subset_pred)
        
        title = f"SNR Range {low}dB to {high}dB (Acc: {subset_acc:.2%})"
        filename = os.path.join(RESULTS_DIR, f"cm_snr_{low}_{high}.png")
        
        plot_confusion_matrix(subset_true, subset_pred, title, filename)
        print(f"  -> Saved {filename} (Acc: {subset_acc:.2%})")

    # --- 6. 真实数据测试 ---
    if len(real_ds) > 0:
        print("\nRunning Real Data Test...")
        real_loader = DataLoader(real_ds, batch_size=BATCH_SIZE, shuffle=False)
        real_true = []
        real_pred = []
        
        with torch.no_grad():
            for imgs, labels, _ in real_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                _, preds = outputs.max(1)
                real_true.extend(labels.cpu().numpy())
                real_pred.extend(preds.cpu().numpy())
        
        real_acc = np.mean(np.array(real_true) == np.array(real_pred))
        print(f"Real Data Accuracy: {real_acc:.4f}")
        
        plot_confusion_matrix(real_true, real_pred, 
                             f"Real Data Confusion Matrix (Acc: {real_acc:.2%})",
                             os.path.join(RESULTS_DIR, "cm_real_data.png"))
        print(f"Results saved to {RESULTS_DIR}")

if __name__ == "__main__":
    main()

Initializing Datasets...
Loading ./dataset_dual_RA\sim_train into RAM...


100%|██████████| 3783/3783 [00:01<00:00, 3019.08img/s]


Loading ./dataset_dual_RA\sim_val into RAM...


100%|██████████| 1017/1017 [00:00<00:00, 3706.44img/s]


Loading ./dataset_dual_RA\real_test into RAM...


100%|██████████| 29292/29292 [00:07<00:00, 3760.38img/s]



Starting Training for 15 epochs...
Epoch 1/15 | Loss: 1.1223 / 1.1184 | Acc: 0.5303 / 0.6303
Epoch 2/15 | Loss: 0.5872 / 0.3717 | Acc: 0.7431 / 0.8850
Epoch 3/15 | Loss: 0.3792 / 0.2392 | Acc: 0.8543 / 0.9272
Epoch 4/15 | Loss: 0.3049 / 0.2283 | Acc: 0.8776 / 0.9125
Epoch 5/15 | Loss: 0.2738 / 0.2000 | Acc: 0.8937 / 0.9204
Epoch 6/15 | Loss: 0.2470 / 0.1908 | Acc: 0.9017 / 0.9194
Epoch 7/15 | Loss: 0.2268 / 0.1701 | Acc: 0.9083 / 0.9263
Epoch 8/15 | Loss: 0.1977 / 0.1527 | Acc: 0.9210 / 0.9292
Epoch 9/15 | Loss: 0.1918 / 0.1742 | Acc: 0.9220 / 0.9213
Epoch 10/15 | Loss: 0.1952 / 0.1641 | Acc: 0.9191 / 0.9272
Epoch 11/15 | Loss: 0.1774 / 0.1359 | Acc: 0.9215 / 0.9361
Epoch 12/15 | Loss: 0.1692 / 0.1555 | Acc: 0.9252 / 0.9282
Epoch 13/15 | Loss: 0.1679 / 0.1438 | Acc: 0.9268 / 0.9341
Epoch 14/15 | Loss: 0.1730 / 0.1490 | Acc: 0.9276 / 0.9312
Epoch 15/15 | Loss: 0.1707 / 0.1393 | Acc: 0.9292 / 0.9331

Running Detailed Evaluation on Validation Set...
Generating Per-SNR Confusion Matrices.